## Imports

In [ ]:
import os
import re
import sys
import json
import ftfy
import unicodedata
import time
import subprocess

from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from tqdm import tqdm
from tqdm.auto import tqdm

# Add root of repo to path to enable module imports
ROOT = Path(os.getcwd(), "..").resolve().as_posix()

if ROOT not in sys.path:
    sys.path.append(ROOT)

from config import Config

## Define key variables

In [ ]:
CONF = Config()

DATA_DIR = Path(ROOT, CONF.find("data.dir"))
DOTENV_PATH = Path(ROOT, "secrets/.env")

TARGET_COLUMNS = [
    "ACL",              # Anterior Cruciate Ligament tear / pathology
    "MCL",              # Medial Collateral Ligament tear / pathology
    "Medial Meniscus",  # Medial Meniscal tear / pathology
    "Lateral Meniscus", # Lateral Meniscal tear / pathology
    "Medial OA",        # Medial Compartment Osteoarthritis
    "Lateral OA",       # Lateral Compartment Osteoarthritis
    "PF OA",            # Patellofemoral Osteoarthritis
    "Effusion",         # Joint Effusion
    "Synovitis",        # Synovitis / Synovial Thickening
    "Baker's",          # Baker's Cyst
    "Contusion",        # Bone Marrow Edema / Contusion
    "Fracture",         # Bone Fracture
]

if load_dotenv(DOTENV_PATH):
    print("Successfully loaded env")
else:
    print("Failed to load env")

## Small EDA of train.csv

In [ ]:
train_df = pd.read_csv(DATA_DIR / CONF.find("data.train"))

In [ ]:
train_df.info()

**There are some duplicate reports**

DONE:
* Verified that for each group of duplicate reports, true labels (if they exist) are consistent.
* Spoke to Alex:
    * If the condition observed is the exact same or if no condition was observed, then reports have no reason to be different.
    * Doctors likely have a standardised way of writing reports.
    * As long as labels are consistent across duplicates (which is true), there is no issue.

TODOs:
* Check for each group of duplicate reports, if the patient id is consistent.

In [ ]:
# keep only duplicates
dup_reports = pd.DataFrame(
    train_df[["StudyInstanceUID", "Report"]][train_df["Report"].duplicated(keep=False)]
)

# col to count the number of times each report appears
dup_reports["Count"] = [1] * len(dup_reports)

# aggregate by report
#   - collate ids into a list
#   - count occurences of each report
agg_dup_reports = dup_reports.groupby(
    by="Report"
).agg(
    {"Count": "sum", "StudyInstanceUID": lambda x: x}
).reset_index( # fix col structure
).rename(
    columns={"StudyInstanceUID": "StudyInstanceUIDs"}
).sort_values(
    by="Count",
    axis=0,
    ascending=False
).reset_index( # correct index after sorting
)[["StudyInstanceUIDs", "Report", "Count"]]

print(agg_dup_reports.describe())
agg_dup_reports.head(10)

^ Looks like one report is duplicated **37 times**!

Analysing the top 5 most frequent reports below
`agg_dup_reports` is sorted, so we can take the report at indexes 0-4

In [ ]:
# print(agg_dup_reports.loc[0, "Report"])
# print(agg_dup_reports.loc[1, "Report"])
# print(agg_dup_reports.loc[2, "Report"])
# print(agg_dup_reports.loc[3, "Report"])
# print(agg_dup_reports.loc[4, "Report"])

## Analysis of 5 most common Reports

*(Report texts and translations removed: competition data cannot be redistributed.)*


Checking if labels are consistent across reports that are the same (if they exist):

In [ ]:
def check_labels_are_consistent(row):
    target_report = row["Report"]
    rows_to_check = train_df[train_df["Report"] == target_report]
    row["are_labels_consistent"] = all(rows_to_check[col].nunique() < 2 for col in TARGET_COLUMNS)
    row["has_labels"] = any(rows_to_check[col].nunique() == 1 for col in TARGET_COLUMNS)
    return row


agg_dup_reports = agg_dup_reports.apply(func=check_labels_are_consistent, axis=1)
agg_dup_reports.head(3)

In [ ]:
if len(agg_dup_reports[agg_dup_reports["are_labels_consistent"] == False]) == 0:
    print("Labels are consistent across Reports which are the same.\n")
else:
    print("[WARNING] Some labels are inconsistent across Reports which are the same.")

In [ ]:
print("Most reports actually don't have labels provided. Here is one which does:")
agg_dup_reports[agg_dup_reports["has_labels"] == True]

## Translating Gold Data (58 rows)

In [ ]:
from google.cloud import translate


def translate_text(
    text="¡Hola Mundo!",
    project_id="translation-v2-506402"
) -> tuple[str, str] | str:
    """
    Returns:
        Tuple of (translated text, detected_language)
    """

    client = translate.TranslationServiceClient()
    location = "global"
    parent = f"projects/{project_id}/locations/{location}"

    try:
        response = client.translate_text(
            request={
                "parent": parent,
                "contents": [text],
                "mime_type": "text/plain",
                "target_language_code": "en",
            }
        )
        # for translation in response.translations:
        #     print("Translated text: {}".format(translation.translated_text))

        translation = response.translations[0]
        return translation.translated_text, translation.detected_language_code

    except Exception as e:
        print(f"[ERROR] Error during translation: {str(e)}")
        return e

In [ ]:
gold_df = pd.read_csv(DATA_DIR / CONF.find("data.gold_label"))
print(f"Row count: {len(gold_df)}")


if len(gold_df[gold_df[TARGET_COLUMNS].isna().any(axis=1)]) == 0:
    pass
else:
    print("[WARNING] There are rows missing labels!")

In [ ]:
gold_translated_path = Path(DATA_DIR / CONF.find("data.gold_translated"))

if gold_translated_path.exists():
    print(f"Loading Translated Labels from {gold_translated_path}")
    gold_translated = pd.read_csv(gold_translated_path)

else:
    gold_translated = gold_df.copy()
    gold_translated.insert(column="Original Language", value=pd.NA, loc=2)
    gold_translated.insert(column="Translated Report", value=pd.NA, loc=3)

    for i in tqdm(range(len(gold_translated))):
        report = gold_translated.loc[i, "Report"]
        translated_report, detected_lang_code = translate_text(report)
        gold_translated.loc[i, "Translated Report"] = translated_report
        gold_translated.loc[i, "Original Language"] = detected_lang_code
        gold_translated.to_csv(gold_translated_path)


## Text Normalisation?

There are a number of non-ASCII characters present in the text including:
* The degree sign °
* U+200B (a zero width space)

BERT cannot handle U+200B characters, so they should be removed.

° is commonly used in medical reports to represent grade which is a measure of severity of a finding

Examples provided below:

### U+200B

In [ ]:
[c for c in gold_translated.loc[6, "Translated Report"] if not c.isascii()]

In [ ]:
print("Zero width space found at position: {}".format(
    gold_translated.loc[6, "Translated Report"].find('\u200b'))
)

In [ ]:
# When printing the \u200b characters are invisible
print(gold_translated.loc[6, "Translated Report"][650:710])

In [ ]:
gold_translated.loc[6, "Translated Report"][650:710]

### ° (the degree)

In [ ]:
gold_translated.loc[41, "Translated Report"].find("°")

In [ ]:
gold_translated.loc[41, "Translated Report"][300:350]

In [ ]:
gold_translated.loc[41, "Translated Report"]

The degree symbol was also used in the original report, so the translation correctly preserved it.

Contextually the degree represents the grade of an observation.

Perhaps we could normalise all findings of degree symbols which also appear next to roman numerals with "Grade \<roman numeral\>"

### A note on translations

I used Claude Sonnet 4.6 to translate the same reports to see how it would handle the degree symbol and how it's wording may differ from google's translation api.

Prompt to Claude:

"
You translate musculoskeletal MRI reports into English.

Render the report using standard English radiology reporting conventions, not word-for-word. Fidelity is to meaning, not to source phrasing.

Conventions:
- The middle segment of a meniscus is the "body", not the "trunk".
- Ligament continuity: "intact" rather than "maintained continuity".
- Phrases meaning normal/unremarkable appearance render as "unremarkable", not "neat presentation".
- Keep the Latin plural "retinacula".

Constraints:
- Add nothing. Omit nothing. Do not hedge, grade, or interpret beyond the source.
- Reproduce every measurement, unit, and number unchanged.
- Preserve laterality (medial/lateral, left/right, anterior/posterior) exactly.
- One output line per input line, same order.
- If a phrase is ambiguous, translate it literally rather than guessing.

Output only the translation. No preamble, no commentary.
"

Here is a comparison:

*(Example report and its two translations removed: competition data cannot be redistributed.)*

<br>
<br>
<br>

KINDA IMPORTANT: In my opinion Claude's wording is better and I think it would also be better for a text encoder. Need to consider re-translating.

## Translating train.csv

In [ ]:
train_translated_path = Path(DATA_DIR / CONF.find("data.train_translated"))

if train_translated_path.exists():
    print(f"Loading Translated Labels from {train_translated_path}")
    train_translated = pd.read_csv(train_translated_path)

else:
    train_translated = train_df.copy()
    train_translated.insert(column="Original Language", value=pd.NA, loc=2)
    train_translated.insert(column="Translated Report", value=pd.NA, loc=3)

    for i in tqdm(range(len(train_translated))):
        report = train_translated.loc[i, "Report"]
        translated_report, detected_lang_code = translate_text(report)
        train_translated.loc[i, "Translated Report"] = translated_report
        train_translated.loc[i, "Original Language"] = detected_lang_code
        train_translated.to_csv(train_translated_path)

## Cleaning translations

In [ ]:
cleaned_translations = train_translated.copy()

# When the original language was english use the original report
cleaned_translations["Translated Report"] = cleaned_translations["Translated Report"].mask(
    cleaned_translations["Original Language"] == "en", cleaned_translations["Report"]
)

In [ ]:
def get_all_non_ascii_chars(df: pd.DataFrame, col: str, log=True):
    result = set()
    not_ascii = re.compile("""[^\x00-\x7F]""")

    for i in range(len(df)):
        non_ascii_chars = not_ascii.findall(
            df.loc[i, col]
        )
        result |= set(non_ascii_chars)
        if non_ascii_chars and log:
            print(i, non_ascii_chars)

    return result

In [ ]:
all_non_ascii_chars = get_all_non_ascii_chars(cleaned_translations, "Translated Report")

In [ ]:
train_translated.loc[3668, "Translated Report"]

In [ ]:
train_translated.loc[3668, "Report"]

In [ ]:
# Issue with RTF parsing
# \\'e0 is parsed to \'e0 
# this is then translated and google translate added the square character..
print(train_translated.loc[3561, "Report"])

In [ ]:
print(all_non_ascii_chars)

In [ ]:
def clean_text(row):
    text = row["Translated Report"]
    fixed = ftfy.fix_text(text)
    norm = unicodedata.normalize('NFKC', fixed)
    norm = norm.replace("\u200b", "").replace("×", "x")
    norm = norm.replace("–", "-").replace("С", "C")
    return norm

cleaned_translations["Cleaned Translations"] = cleaned_translations.apply(func=clean_text, axis=1)

In [ ]:
cleaned_non_ascii_chars = get_all_non_ascii_chars(cleaned_translations, "Cleaned Translations")

In [ ]:
print(cleaned_non_ascii_chars)

In [ ]:
# The 'τ' here comes from the original and google translate left it in..
cleaned_translations.loc[3214, "Cleaned Translations"]

In [ ]:
cleaned_translations.loc[3214, "Report"]

In [ ]:
# (Translated report removed: competition data cannot be redistributed.)


## Re-translation is probably a good idea

^ I think all of the above indicates retranslation using an LLM is probably a smart idea.

It is also clear that some preprocessing of the original text should be done to handle RTF issues

Would probably save time having to go through and clean the original data also..

In [ ]:
cleaned_translations_path = Path(DATA_DIR / CONF.find("data.train_translated_clean"))

cleaned_translations.to_csv(cleaned_translations_path)

## Using Claude code for translations

In [ ]:
CHECKPOINT = Path(DATA_DIR / CONF.find("data.claude_translations"))

FAILED_LOG = Path("translations_failed.jsonl")
MODEL = "opus"
TIMEOUT_S = 180
MAX_ATTEMPTS = 3
CHECKPOINT_EVERY = 1

SYSTEM_PROMPT= """
You translate musculoskeletal MRI reports into English.

Render the report using standard English radiology reporting conventions, not word-for-word. Fidelity is to meaning, not to source phrasing.

Conventions:
- The middle segment of a meniscus is the "body", not the "trunk".
- Ligament continuity: "intact" rather than "maintained continuity".
- Phrases meaning normal/unremarkable appearance render as "unremarkable", not "neat presentation".
- Keep the Latin plural "retinacula".

Constraints:
- Add nothing. Omit nothing. Do not hedge, grade, or interpret beyond the source.
- Reproduce every measurement, unit, and number unchanged.
- Preserve laterality (medial/lateral, left/right, anterior/posterior) exactly.
- One output line per input line, same order.
- If a phrase is ambiguous, translate it literally rather than guessing.

Output only the translation. No preamble, no commentary.
"""

In [ ]:
def call_claude(prompt: str) -> str:
    """Run one prompt through `claude -p` and return the result text.
 
    Raises RuntimeError on non-zero exit or unparseable output so the
    caller can decide whether to retry.
    """
    cmd = [
        "claude", "-p", prompt,
        "--model", MODEL,
        "--output-format", "json",
        "--max-turns", "1",
        "--system-prompt", SYSTEM_PROMPT,
        "--tools", "",
        "--mcp-config", '{"mcpServers":{}}',
        "--strict-mcp-config",
    ]
    proc = subprocess.run(
        cmd, capture_output=True, text=True, timeout=TIMEOUT_S,
        stdin=subprocess.DEVNULL,
    )
    if proc.returncode != 0:
        raise RuntimeError(f"exit {proc.returncode}: {proc.stderr.strip()[:500]}")
    try:
        payload = json.loads(proc.stdout)
    except json.JSONDecodeError as e:
        raise RuntimeError(f"bad JSON: {e}: {proc.stdout[:300]}")
    if payload.get("is_error"):
        raise RuntimeError(f"claude reported error: {payload.get('result')}")
    return payload["result"]

def call_with_retry(prompt: str) -> str:
    """Retry with exponential backoff; re-raise after MAX_ATTEMPTS."""
    last_err = None
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            return call_claude(prompt)
        except (RuntimeError, subprocess.TimeoutExpired) as e:
            last_err = e
            wait = 5 * 2 ** (attempt - 1)   # 5s, 10s, 20s
            print(f"  attempt {attempt} failed ({str(e)[:120]}), waiting {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"gave up after {MAX_ATTEMPTS} attempts: {last_err}")
 

In [ ]:
def run_batch(df: pd.DataFrame, source_col: str, out_col: str = "LLM Translation"):
    # resume from checkpoint if present
    if CHECKPOINT.exists():
        done = pd.read_csv(CHECKPOINT, index_col=0)
        df = df.copy()
        df[out_col] = done[out_col].reindex(df.index).astype("object")
        print(f"resumed: {df[out_col].notna().sum()} rows already translated")
    elif out_col not in df:
        df = df.copy()
        df[out_col] = pd.Series(pd.NA, index=df.index, dtype="object")
 
    todo = df.index[df[out_col].isna()]
    print(f"{len(todo)} rows to do")

    bar = tqdm(todo, desc="translating", unit="row")
    n_failed = 0
    for n, idx in enumerate(bar, start=1):
        source = df.at[idx, source_col]
        if not isinstance(source, str) or not source.strip():
            df.at[idx, out_col] = ""
            continue

        prompt = f"Translate this report into standard English:\n\n{source}"
        try:
            df.at[idx, out_col] = call_with_retry(prompt)
        except RuntimeError as e:
            n_failed += 1
            with FAILED_LOG.open("a") as f:
                f.write(json.dumps({"index": int(idx), "error": str(e)}) + "\n")
            df.at[idx, out_col] = pd.NA

        bar.set_postfix(failed=n_failed, last=idx)

        if n % CHECKPOINT_EVERY == 0 or n == len(todo):
            df.to_csv(CHECKPOINT)
            print(f"[{n}/{len(todo)}] checkpoint saved")
 
    return df

In [ ]:
cleaned_translations = run_batch(cleaned_translations, source_col="Report")

In [ ]:
cleaned_translations.to_csv(CHECKPOINT)

In [ ]:
cleaned_llm_non_ascii_chars = get_all_non_ascii_chars(cleaned_translations, "LLM Translation")

In [ ]:
cleaned_llm_non_ascii_chars

### What are all these non-ascii characters..?

I am not touching these non ascii chars for now. I think it best to handle them when preprocessing for a particular text encoder.

* "–" en dash
* "—" em dash
* "“", "”" quotation marks
* "•" bullet point
* "½"
    * extract from StudyInstanceUID = study-b209cece07cc078f
    * (report excerpt removed)
* "²"
    * extract from StudyInstanceUID = study-252783d3963e2da4
    * (report excerpt removed)
* "×"
    * extract from StudyInstanceUID = study-2e5400bcf099e9a7
    * (report excerpt removed)
* "°" used in context of angles (manually changed when used in grading context) We could therefore replace this with ' degrees' in preprocessing for the text encoder.
    * extract from StudyInstanceUID = study-cd2e6ee9829aa50b
    * (report excerpt removed)
* "ä"
    * appears with mention of "Ahlbäck disease"
* "é"
    * appears with mentions of "Morel-Lavallée" which is a type of injury.